In [1]:
import re
import copy
import torch
import warnings
import numpy as np
import pandas as pd
import torch.nn as nn
from copy import deepcopy
from sklearn.base import clone
import torch.nn.functional as F
warnings.filterwarnings('ignore')
from xgboost import XGBClassifier
from torch_geometric.data import HeteroData
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from torch_geometric.nn import GCNConv, HeteroConv, GATv2Conv, GraphNorm
from sklearn.metrics import ( accuracy_score, precision_score, recall_score, 
    f1_score, classification_report, confusion_matrix )

# Paths
DATA_PATH = "Incidents_imputed.xlsx"
GRAPH_PATH = "Hetro_Final_NW_graph_1.pt"

incident_df = pd.read_excel(DATA_PATH, parse_dates=['Job OFF Time', 'Job ON Time'])
hetero_graph = torch.load(GRAPH_PATH)

print(f"Loaded incidents: {len(incident_df)}")
print("Loaded Graph Structure:")
print(hetero_graph)



Loaded incidents: 292829
Loaded Graph Structure:
HeteroData(
  substation={
    x=[347, 8],
    node_ids=[347],
  },
  (substation, spatial, substation)={
    edge_index=[2, 23378],
    edge_attr=[23378, 8],
  },
  (substation, temporal, substation)={
    edge_index=[2, 37924],
    edge_attr=[37924, 2],
  },
  (substation, causal, substation)={
    edge_index=[2, 10136],
    edge_attr=[10136, 13],
  }
)


In [2]:
####### Maintenance Target with days window

def create_graph_with_target_window(graph, incident_df, day_window):
    """
    Cleans incident data, computes target labels based on a custom day window,
    and integrates them into a deep copy of the provided HeteroData graph.
    """
    # Deep copy to preserve original
    graph_copy = copy.deepcopy(graph)
    valid_substations = set(graph_copy['substation'].node_ids)

    # --- Clean Substation Names ---
    df = incident_df.copy()
    def clean_substation_name(x):
        match = re.match(r'(\d+)\s*:\s*(.+)', str(x).strip().upper())
        return f"{match.group(1)}:{match.group(2).strip()}" if match else None
    df['Job Substation'] = df['Job Substation'].astype(str).apply(clean_substation_name)
    dropped_count = df['Job Substation'].isna().sum()
    df.dropna(subset=['Job Substation'], inplace=True)

    # --- Severe Causes ---
    cause_stats = df.groupby('Cause Desc').agg(
        median_duration=('Job Duration Mins', 'median'),
        max_customers=('Custs Affected', 'max')
    )
    cause_thresholds = cause_stats.quantile(0.90)
    severe_causes = cause_stats[
        (cause_stats['median_duration'] > cause_thresholds['median_duration']) &
        (cause_stats['max_customers'] > cause_thresholds['max_customers'])
    ].index.unique()

    # --- Critical Equipment ---
    equip_stats = df.groupby('Equip Desc').agg(
        failure_freq=('Job Display ID', 'count'),
        saidi=('Job SAIDI', 'mean')
    )
    equipment_thresholds = equip_stats.quantile(0.90)
    critical_equipment = df.groupby('Equip Desc').filter(
        lambda x: (len(x) > equipment_thresholds['failure_freq']) and
                  (x['Job SAIDI'].mean() > equipment_thresholds['saidi'])
    )['Equip Desc'].unique()

    # --- Compute Target Variable ---
    df_sorted = df.sort_values(['Job Substation', 'Equip Desc', 'Job OFF Time']).copy()
    df_sorted['days_until_next_incident'] = df_sorted.groupby(
        ['Job Substation', 'Equip Desc']
    )['Job OFF Time'].diff(-1).dt.days.abs()

    df_sorted['needs_replacement'] = np.where(
        (df_sorted['Cause Desc'].isin(severe_causes)) & 
        (df_sorted['Equip Desc'].isin(critical_equipment)) & 
        ((df_sorted['days_until_next_incident'] > day_window) | 
         df_sorted['days_until_next_incident'].isna()),
        1,
        0
    )

    final_targets = df_sorted.groupby('Job Substation')['needs_replacement'].max()
    final_targets = final_targets[final_targets.index.isin(valid_substations)]

    # --- Integrate Target into Graph ---
    substation_names = graph_copy['substation'].node_ids
    final_targets = final_targets.reindex(substation_names).fillna(0)
    graph_copy['substation'].y = torch.tensor(final_targets.values.astype(np.float32), dtype=torch.float)

    assert len(graph_copy['substation'].y) == len(substation_names), "Mismatch in target integration!"

    # Debugging Info
    print(f"\n Day Window: {day_window}")
    print(f"Dropped {dropped_count} invalid substations.")
    print(f"Targets integrated: {int(final_targets.sum())} positive out of {len(final_targets)} substations.")
    print("Target distribution:")
    print(final_targets.value_counts())

    return graph_copy

# === Create graphs with multiple day windows ===
hetero_graph_30 = create_graph_with_target_window(hetero_graph, incident_df, day_window=30)
hetero_graph_60 = create_graph_with_target_window(hetero_graph, incident_df, day_window=60)
hetero_graph_180 = create_graph_with_target_window(hetero_graph, incident_df, day_window=180)



 Day Window: 30
Dropped 2 invalid substations.
Targets integrated: 197 positive out of 347 substations.
Target distribution:
needs_replacement
1    197
0    150
Name: count, dtype: int64

 Day Window: 60
Dropped 2 invalid substations.
Targets integrated: 191 positive out of 347 substations.
Target distribution:
needs_replacement
1    191
0    156
Name: count, dtype: int64

 Day Window: 180
Dropped 2 invalid substations.
Targets integrated: 170 positive out of 347 substations.
Target distribution:
needs_replacement
0    177
1    170
Name: count, dtype: int64


In [ ]:
############## Classical ML with 30, 60, 180 days window #########

# --- Helpers ---
def clean_substation_name(x):
    import re
    match = re.match(r'(\d+)\s*:\s*(.+)', str(x).strip().upper())
    return f"{match.group(1)}:{match.group(2).strip()}" if match else None


def create_target_variable_foldwise(df, cutoff_date, valid_substations=None, day_window=180):
    """Fold-specific target creation using only data up to cutoff_date"""
    df = df[df['Job OFF Time'] <= cutoff_date].copy()
    df['Job Substation'] = df['Job Substation'].astype(str).apply(clean_substation_name)
    df.dropna(subset=['Job Substation'], inplace=True)

    cause_stats = df.groupby('Cause Desc').agg(
        median_duration=('Job Duration Mins', 'median'),
        max_customers=('Custs Affected', 'max')
    )
    cause_thresholds = cause_stats.quantile(0.90)
    severe_causes = cause_stats[
        (cause_stats['median_duration'] > cause_thresholds['median_duration']) &
        (cause_stats['max_customers'] > cause_thresholds['max_customers'])
    ].index.unique()

    equip_stats = df.groupby('Equip Desc').agg(
        failure_freq=('Job Display ID', 'count'),
        saidi=('Job SAIDI', 'mean')
    )
    equipment_thresholds = equip_stats.quantile(0.90)
    critical_equipment = df.groupby('Equip Desc').filter(
        lambda x: (len(x) > equipment_thresholds['failure_freq']) and
                  (x['Job SAIDI'].mean() > equipment_thresholds['saidi'])
    )['Equip Desc'].unique()

    df_sorted = df.sort_values(['Job Substation', 'Equip Desc', 'Job OFF Time']).copy()
    df_sorted['days_until_next_incident'] = df_sorted.groupby(
        ['Job Substation', 'Equip Desc']
    )['Job OFF Time'].diff(-1).dt.days.abs()

    # 💡 Use day_window here
    df_sorted['needs_replacement'] = np.where(
        (df_sorted['Cause Desc'].isin(severe_causes)) &
        (df_sorted['Equip Desc'].isin(critical_equipment)) &
        ((df_sorted['days_until_next_incident'] > day_window) |
         df_sorted['days_until_next_incident'].isna()),
        1, 0
    )

    final_targets = df_sorted.groupby('Job Substation')['needs_replacement'].max()

    if valid_substations is not None:
        final_targets = final_targets[final_targets.index.isin(valid_substations)]

    return final_targets


def create_clean_node_features_from_graph(hetero_graph, incident_df, cutoff_date):
    """
    Uses node features already stored in hetero_graph but ensures no future data is leaked.
    Filters incident data to only include incidents up to cutoff_date,
    and returns features aligned with node_ids.
    """
    node_ids = hetero_graph['substation'].node_ids
    node_features_tensor = hetero_graph['substation'].x
    node_features_df = pd.DataFrame(node_features_tensor.cpu().numpy(), index=node_ids)

    # Filter incident data to avoid leakage
    filtered_incidents = incident_df[incident_df['Job OFF Time'] <= cutoff_date].copy()
    filtered_incidents['Job Substation'] = filtered_incidents['Job Substation'].astype(str).apply(clean_substation_name)

    # Restrict node features to substations present in incident_df (up to this fold)
    valid_nodes = set(filtered_incidents['Job Substation'].unique()).intersection(set(node_ids))
    node_features_df = node_features_df.reindex(index=node_ids).fillna(0)

    return node_features_df


def leakage_aware_evaluation_leakage_proof_from_graph(incident_df, hetero_graph, n_splits=5, day_window=180):
    """Leakage-proof evaluation using only safe node features from hetero_graph and incident-based targets"""
    results = {}

    node_ids = hetero_graph['substation'].node_ids
    node_list = list(node_ids)
    valid_substations = set(node_list)

    full_targets = create_target_variable_foldwise(
        incident_df, cutoff_date=pd.Timestamp.max,
        valid_substations=valid_substations,
        day_window=day_window  # ✅ Use it here
    )

    X_all = np.arange(len(node_list))
    y_all = full_targets.reindex(node_list).fillna(0).astype(int).values

    models = {
        "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                                max_depth=5, random_state=42),
        "XGBoost": XGBClassifier(n_estimators=100, scale_pos_weight=np.sum(y_all == 0) / np.sum(y_all == 1),
                                 max_depth=3, use_label_encoder=False, eval_metric='logloss', random_state=42)
    }

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for model_name, model in models.items():
        fold_metrics = []

        for fold, (train_idx, test_idx) in enumerate(skf.split(X_all, y_all)):
            train_subs = [node_list[i] for i in train_idx]
            test_subs = [node_list[i] for i in test_idx]

            train_incidents = incident_df[incident_df['Job Substation'].isin(train_subs)]
            last_train_date = train_incidents['Job OFF Time'].max()

            # Features change per fold
            X_df = create_clean_node_features_from_graph(
                hetero_graph=hetero_graph,
                incident_df=incident_df,
                cutoff_date=last_train_date
            )

            # ✅ Use custom day window
            y_fold = create_target_variable_foldwise(
                incident_df, cutoff_date=last_train_date,
                valid_substations=set(X_df.index),
                day_window=day_window
            )

            X_fold = X_df.reindex(node_list).fillna(0).values
            y_fold = y_fold.reindex(node_list).fillna(0).astype(int).values

            X_train = X_fold[train_idx]
            X_test = X_fold[test_idx]
            y_train = y_fold[train_idx]
            y_test = y_fold[test_idx]

            fold_model = clone(model)
            fold_model.fit(X_train, y_train)
            preds = fold_model.predict(X_test)

            metrics = {
                'accuracy': accuracy_score(y_test, preds),
                'precision': precision_score(y_test, preds, zero_division=0),
                'recall': recall_score(y_test, preds, zero_division=0),
                'f1': f1_score(y_test, preds),
                'report': classification_report(y_test, preds, output_dict=True),
                'confusion': confusion_matrix(y_test, preds)
            }

            fold_metrics.append(metrics)

        results[model_name] = fold_metrics

    return results


for days in [180, 60, 30]:
    print(f"\n{'#' * 15} {days}-DAY EVALUATION {'#' * 15}")
    results = leakage_aware_evaluation_leakage_proof_from_graph(
        incident_df=incident_df,
        hetero_graph=hetero_graph,
        n_splits=5,
        day_window=days  # 👈 This changes the logic dynamically!
    )
    summarize_metrics(results)



In [ ]:
# Save classical ML evaluation results to text file
with open("final_classical_ml_results.txt", "w") as f:
    for days in [180, 60, 30]:
        f.write(f"\n{'#' * 15} {days}-DAY EVALUATION {'#' * 15}\n")

        results = leakage_aware_evaluation_leakage_proof_from_graph(
            incident_df=incident_df,
            hetero_graph=hetero_graph,
            n_splits=5,
            day_window=days
        )

        for model_name, folds in results.items():
            f.write(f"\n===== {model_name} (Mean ± Std) =====\n")
            for metric in ['accuracy', 'precision', 'recall', 'f1']:
                values = [fold[metric] for fold in folds]
                mean = np.mean(values)
                std = np.std(values)
                f.write(f"{metric.capitalize():<10}: {mean:.4f} ± {std:.4f}\n")


In [6]:
############ Final Multi_layer_GNN Vs Single Layer


# Hyperparameter grid
param_grid = {
    'hidden': [64, 128],
    'heads': [2, 4],
    'dropout': [0.3, 0.5],
    'lr': [1e-3, 1e-4],
    'weight_decay': [1e-4, 1e-5]
}

# Define the model
class EnhancedHeteroGNN(torch.nn.Module):
    def __init__(self, hidden=128, heads=4, dropout=0.3, edge_types=None):
        super().__init__()
        self.hidden = hidden
        self.heads = heads
        self.dropout = dropout
        self.edge_types = edge_types

        self.node_encoder = torch.nn.Linear(8, hidden)

        # Edge feature encoders
        self.edge_encoders = torch.nn.ModuleDict({
            'spatial': torch.nn.Linear(8, hidden // 2),
            'temporal': torch.nn.Linear(2, hidden // 2),
            'causal': torch.nn.Linear(13, hidden // 2),
        })

        self.conv1 = HeteroConv({
            etype: GATv2Conv(hidden, hidden, heads=heads)
            for etype in edge_types
        }, aggr='mean')

        self.norm1 = GraphNorm(hidden * heads)

        self.conv2 = HeteroConv({
            etype: GATv2Conv(hidden * heads, hidden, heads=1)
            for etype in edge_types
        }, aggr='mean')

        self.classifier = torch.nn.Linear(hidden, 2)

    def forward(self, data):
        x = self.node_encoder(data['substation'].x)

        edge_attrs = {
            etype[1]: self.edge_encoders[etype[1]](data[etype].edge_attr)
            for etype in self.edge_types
        }

        x_dict = self.conv1({'substation': x}, data.edge_index_dict, edge_attrs)
        x = F.elu(self.norm1(x_dict['substation']))
        x = F.dropout(x, p=self.dropout, training=self.training)

        x_dict = self.conv2({'substation': x}, data.edge_index_dict)
        x = F.elu(x_dict['substation'])

        return F.log_softmax(self.classifier(x), dim=1)


# Main Evaluation Function
def evaluate_gnn_configs(hetero_graph, param_grid, k_folds=3):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    edge_configs = {
        "all": [
            ('substation', 'spatial', 'substation'),
            ('substation', 'temporal', 'substation'),
            ('substation', 'causal', 'substation')
        ],
        "spatial_only": [('substation', 'spatial', 'substation')],
        "temporal_only": [('substation', 'temporal', 'substation')],
        "causal_only": [('substation', 'causal', 'substation')]
    }

    y = hetero_graph['substation'].y.numpy().flatten().astype(int)
    class_counts = np.bincount(y)
    class_weights = torch.tensor([1.0 / c for c in class_counts], dtype=torch.float32).to(device)

    hetero_graph.edge_index_dict = {
        etype: hetero_graph[etype].edge_index for etype in hetero_graph.edge_types
    }

    results = {}

    for config_name, edge_types in edge_configs.items():
        print(f"\n🔍 Evaluating config: {config_name.upper()}")
        best_f1 = -1
        best_params = None
        best_metrics = None

        for params in ParameterGrid(param_grid):
            #print(f" → Trying params: {params}")
            fold_metrics = []

            skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
            for fold, (train_idx, test_idx) in enumerate(skf.split(np.arange(len(y)), y)):
                model = EnhancedHeteroGNN(
                    hidden=params['hidden'],
                    heads=params['heads'],
                    dropout=params['dropout'],
                    edge_types=edge_types
                ).to(device)

                data = deepcopy(hetero_graph).to(device)
                optimizer = torch.optim.AdamW(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])

                train_mask = torch.zeros(len(y), dtype=torch.bool)
                train_mask[train_idx] = True
                test_mask = torch.zeros(len(y), dtype=torch.bool)
                test_mask[test_idx] = True

                # Training
                best_fold_f1 = 0
                for epoch in range(100):
                    model.train()
                    optimizer.zero_grad()
                    out = model(data)
                    loss = F.nll_loss(out[train_mask], data['substation'].y[train_mask].long(), weight=class_weights)
                    loss.backward()
                    optimizer.step()

                # Evaluation
                model.eval()
                with torch.no_grad():
                    pred = model(data).argmax(dim=1)
                    y_true = data['substation'].y[test_mask].cpu().numpy()
                    y_pred = pred[test_mask].cpu().numpy()

                    acc = accuracy_score(y_true, y_pred)
                    prec = precision_score(y_true, y_pred, zero_division=0)
                    rec = recall_score(y_true, y_pred, zero_division=0)
                    f1 = f1_score(y_true, y_pred)

                    fold_metrics.append((acc, prec, rec, f1))

            # Avg metrics
            accs, precs, recs, f1s = zip(*fold_metrics)
            avg_f1 = np.mean(f1s)

            if avg_f1 > best_f1:
                best_f1 = avg_f1
                best_params = params
                best_metrics = {
                    'accuracy': (np.mean(accs), np.std(accs)),
                    'precision': (np.mean(precs), np.std(precs)),
                    'recall': (np.mean(recs), np.std(recs)),
                    'f1': (np.mean(f1s), np.std(f1s))
                }

        # Store best for this config
        results[config_name] = {
            'best_params': best_params,
            'metrics': best_metrics
        }

    # Print summary
    print("\n🎯 FINAL RESULTS SUMMARY")
    for config, data in results.items():
        print(f"\nConfig: {config.upper()}")
        print(f"Best Params: {data['best_params']}")
        for metric, (mean, std) in data['metrics'].items():
            print(f"{metric.capitalize():<10}: {mean:.4f} ± {std:.4f}")

    return results


# Run evaluation on all three datasets
all_graphs = {
    "180-day": hetero_graph_180,
    "60-day": hetero_graph_60,
    "30-day": hetero_graph_30
}

final_all_results = {}

for label, graph in all_graphs.items():
    print(f"\n{'=' * 25} {label.upper()} GNN EVALUATION {'=' * 25}")
    results = evaluate_gnn_configs(graph, param_grid, k_folds=3)
    final_all_results[label] = results

# Clean final summary
print("\n\n" + "#" * 40 + " FINAL BEST GNN RESULTS " + "#" * 40)
for label, configs in final_all_results.items():
    print(f"\n📊 {label.upper()}")
    for config_name, result in configs.items():
        print(f"  → Config: {config_name.upper()}")
        for metric, (mean, std) in result['metrics'].items():
            print(f"    {metric.capitalize():<10}: {mean:.4f} ± {std:.4f}")
        print(f"    Best Params: {result['best_params']}")



========================= 180-DAY GNN EVALUATION =========================

🔍 Evaluating config: ALL

🔍 Evaluating config: SPATIAL_ONLY

🔍 Evaluating config: TEMPORAL_ONLY

🔍 Evaluating config: CAUSAL_ONLY

🎯 FINAL RESULTS SUMMARY

Config: ALL
Best Params: {'dropout': 0.3, 'heads': 2, 'hidden': 64, 'lr': 0.0001, 'weight_decay': 0.0001}
Accuracy  : 0.8386 ± 0.0145
Precision : 0.8140 ± 0.0200
Recall    : 0.8704 ± 0.0308
F1        : 0.8408 ± 0.0147

Config: SPATIAL_ONLY
Best Params: {'dropout': 0.5, 'heads': 2, 'hidden': 64, 'lr': 0.0001, 'weight_decay': 0.0001}
Accuracy  : 0.6311 ± 0.0190
Precision : 0.5807 ± 0.0135
Recall    : 0.8881 ± 0.0230
F1        : 0.7022 ± 0.0169

Config: TEMPORAL_ONLY
Best Params: {'dropout': 0.5, 'heads': 2, 'hidden': 128, 'lr': 0.001, 'weight_decay': 0.0001}
Accuracy  : 0.8214 ± 0.0384
Precision : 0.8075 ± 0.0410
Recall    : 0.8354 ± 0.0325
F1        : 0.8212 ± 0.0368

Config: CAUSAL_ONLY
Best Params: {'dropout': 0.3, 'heads': 4, 'hidden': 128, 'lr': 0.0001, 

In [ ]:
# Save final summary to a text file
with open("final_best_gnn_results.txt", "w") as f:
    f.write("#" * 40 + " FINAL BEST GNN RESULTS " + "#" * 40 + "\n")
    for label, configs in final_all_results.items():
        f.write(f"\n📊 {label.upper()}\n")
        for config_name, result in configs.items():
            f.write(f"  → Config: {config_name.upper()}\n")
            for metric, (mean, std) in result['metrics'].items():
                f.write(f"    {metric.capitalize():<10}: {mean:.4f} ± {std:.4f}\n")
            f.write(f"    Best Params: {result['best_params']}\n")


In [3]:
##### Overleaf reported model

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, GATv2Conv, GINConv, GraphNorm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import StratifiedKFold
from copy import deepcopy
import numpy as np
from sklearn.model_selection import ParameterGrid

# Hyperparameter grid
param_grid = {
    'hidden': [64, 128],
    'heads': [2, 4],
    'dropout': [0.3, 0.5],
    'lr': [1e-3, 1e-4],
    'weight_decay': [1e-4, 1e-5]
}

# Define the revised model
class EnhancedHeteroGNN(torch.nn.Module):
    def __init__(self, hidden=128, heads=4, dropout=0.3, edge_types=None):
        super().__init__()
        self.hidden = hidden
        self.heads = heads
        self.dropout = dropout
        self.edge_types = edge_types

        self.node_encoder = torch.nn.Linear(8, hidden)

        # Edge feature encoders
        self.edge_encoders = torch.nn.ModuleDict({
            'spatial': torch.nn.Linear(8, hidden // 2),
            'temporal': torch.nn.Linear(2, hidden // 2),
            'causal': torch.nn.Linear(13, hidden // 2),
        })

        # First convolutional layer with GATv2 for spatial/temporal and GIN for causal
        conv1_dict = {}
        for etype in edge_types:
            if etype[1] in ['spatial', 'temporal']:
                conv1_dict[etype] = GATv2Conv(hidden, hidden, heads=heads, edge_dim=hidden // 2)
            elif etype[1] == 'causal':
                conv1_dict[etype] = GINConv(nn=nn.Linear(hidden, hidden * heads), train_eps=True)
        self.conv1 = HeteroConv(conv1_dict, aggr='mean')

        self.norm1 = GraphNorm(hidden * heads)

        # Second convolutional layer with GATv2 for spatial/temporal and GIN for causal
        conv2_dict = {}
        for etype in edge_types:
            if etype[1] in ['spatial', 'temporal']:
                conv2_dict[etype] = GATv2Conv(hidden * heads, hidden, heads=1)
            elif etype[1] == 'causal':
                conv2_dict[etype] = GINConv(nn=nn.Linear(hidden * heads, hidden), train_eps=True)
        self.conv2 = HeteroConv(conv2_dict, aggr='mean')

        self.classifier = torch.nn.Linear(hidden, 2)

    def forward(self, data):
        x = self.node_encoder(data['substation'].x)

        edge_attrs = {
            etype[1]: self.edge_encoders[etype[1]](data[etype].edge_attr)
            for etype in self.edge_types if etype[1] in self.edge_encoders
        }

        x_dict = self.conv1({'substation': x}, data.edge_index_dict, edge_attrs)
        x = F.elu(self.norm1(x_dict['substation']))
        x = F.dropout(x, p=self.dropout, training=self.training)

        x_dict = self.conv2({'substation': x}, data.edge_index_dict)
        x = F.elu(x_dict['substation'])

        return F.log_softmax(self.classifier(x), dim=1)


# Main Evaluation Function
def evaluate_gnn_configs(hetero_graph, param_grid, k_folds=3):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    edge_configs = {
        "all": [
            ('substation', 'spatial', 'substation'),
            ('substation', 'temporal', 'substation'),
            ('substation', 'causal', 'substation')
        ],
        "spatial_only": [('substation', 'spatial', 'substation')],
        "temporal_only": [('substation', 'temporal', 'substation')],
        "causal_only": [('substation', 'causal', 'substation')]
    }

    y = hetero_graph['substation'].y.numpy().flatten().astype(int)
    class_counts = np.bincount(y)
    class_weights = torch.tensor([1.0 / c for c in class_counts], dtype=torch.float32).to(device)

    hetero_graph.edge_index_dict = {
        etype: hetero_graph[etype].edge_index for etype in hetero_graph.edge_types
    }

    results = {}

    for config_name, edge_types in edge_configs.items():
        print(f"\n🔍 Evaluating config: {config_name.upper()}")
        best_f1 = -1
        best_params = None
        best_metrics = None

        for params in ParameterGrid(param_grid):
            #print(f" → Trying params: {params}")
            fold_metrics = []

            skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
            for fold, (train_idx, test_idx) in enumerate(skf.split(np.arange(len(y)), y)):
                model = EnhancedHeteroGNN(
                    hidden=params['hidden'],
                    heads=params['heads'],
                    dropout=params['dropout'],
                    edge_types=edge_types
                ).to(device)

                data = deepcopy(hetero_graph).to(device)
                optimizer = torch.optim.AdamW(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])

                train_mask = torch.zeros(len(y), dtype=torch.bool)
                train_mask[train_idx] = True
                test_mask = torch.zeros(len(y), dtype=torch.bool)
                test_mask[test_idx] = True

                # Training
                best_fold_f1 = 0
                for epoch in range(100):
                    model.train()
                    optimizer.zero_grad()
                    out = model(data)
                    loss = F.nll_loss(out[train_mask], data['substation'].y[train_mask].long(), weight=class_weights)
                    loss.backward()
                    optimizer.step()

                # Evaluation
                model.eval()
                with torch.no_grad():
                    pred = model(data).argmax(dim=1)
                    y_true = data['substation'].y[test_mask].cpu().numpy()
                    y_pred = pred[test_mask].cpu().numpy()

                    acc = accuracy_score(y_true, y_pred)
                    prec = precision_score(y_true, y_pred, zero_division=0)
                    rec = recall_score(y_true, y_pred, zero_division=0)
                    f1 = f1_score(y_true, y_pred)

                    fold_metrics.append((acc, prec, rec, f1))

            # Avg metrics
            accs, precs, recs, f1s = zip(*fold_metrics)
            avg_f1 = np.mean(f1s)

            if avg_f1 > best_f1:
                best_f1 = avg_f1
                best_params = params
                best_metrics = {
                    'accuracy': (np.mean(accs), np.std(accs)),
                    'precision': (np.mean(precs), np.std(precs)),
                    'recall': (np.mean(recs), np.std(recs)),
                    'f1': (np.mean(f1s), np.std(f1s))
                }

        # Store best for this config
        results[config_name] = {
            'best_params': best_params,
            'metrics': best_metrics
        }

    # Print summary
    print("\n🎯 FINAL RESULTS SUMMARY")
    for config, data in results.items():
        print(f"\nConfig: {config.upper()}")
        print(f"Best Params: {data['best_params']}")
        for metric, (mean, std) in data['metrics'].items():
            print(f"{metric.capitalize():<10}: {mean:.4f} ± {std:.4f}")

    return results


# Run evaluation on all three datasets
all_graphs = {
    "180-day": hetero_graph_180,
    "60-day": hetero_graph_60,
    "30-day": hetero_graph_30
}

final_all_results = {}

for label, graph in all_graphs.items():
    print(f"\n{'=' * 25} {label.upper()} GNN EVALUATION {'=' * 25}")
    results = evaluate_gnn_configs(graph, param_grid, k_folds=3)
    final_all_results[label] = results

# Clean final summary
print("\n\n" + "#" * 40 + " FINAL BEST GNN RESULTS " + "#" * 40)
for label, configs in final_all_results.items():
    print(f"\n📊 {label.upper()}")
    for config_name, result in configs.items():
        print(f"  → Config: {config_name.upper()}")
        for metric, (mean, std) in result['metrics'].items():
            print(f"    {metric.capitalize():<10}: {mean:.4f} ± {std:.4f}")
        print(f"    Best Params: {result['best_params']}")


========================= 180-DAY GNN EVALUATION =========================

🔍 Evaluating config: ALL

🔍 Evaluating config: SPATIAL_ONLY

🔍 Evaluating config: TEMPORAL_ONLY

🔍 Evaluating config: CAUSAL_ONLY

🎯 FINAL RESULTS SUMMARY

Config: ALL
Best Params: {'dropout': 0.3, 'heads': 4, 'hidden': 128, 'lr': 0.001, 'weight_decay': 1e-05}
Accuracy  : 0.7694 ± 0.0168
Precision : 0.8104 ± 0.0567
Recall    : 0.7053 ± 0.0816
F1        : 0.7481 ± 0.0281

Config: SPATIAL_ONLY
Best Params: {'dropout': 0.5, 'heads': 2, 'hidden': 128, 'lr': 0.001, 'weight_decay': 0.0001}
Accuracy  : 0.6829 ± 0.0298
Precision : 0.6459 ± 0.0162
Recall    : 0.7759 ± 0.0811
F1        : 0.7037 ± 0.0420

Config: TEMPORAL_ONLY
Best Params: {'dropout': 0.5, 'heads': 4, 'hidden': 128, 'lr': 0.0001, 'weight_decay': 1e-05}
Accuracy  : 0.8156 ± 0.0037
Precision : 0.7949 ± 0.0133
Recall    : 0.8412 ± 0.0144
F1        : 0.8171 ± 0.0003

Config: CAUSAL_ONLY
Best Params: {'dropout': 0.5, 'heads': 2, 'hidden': 64, 'lr': 0.0001, 'w